In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

19:26:01 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "drd2",
    "fp_type":     "rdkit_desc",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

19:26:01 [INFO] Loaded lo/drd2 fold 1: train=2206, test=267
19:26:01 [INFO] Fold 1 | train=2206 y_mean=6.721 y_std=0.867 | test=267 n_clusters=50
19:26:01 [INFO] Loaded lo/drd2 fold 2: train=2128, test=267
19:26:01 [INFO] Fold 2 | train=2128 y_mean=6.736 y_std=0.840 | test=267 n_clusters=50
19:26:01 [INFO] Loaded lo/drd2 fold 3: train=2257, test=262
19:26:01 [INFO] Fold 3 | train=2257 y_mean=6.753 y_std=0.857 | test=262 n_clusters=50
19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_1.npz
19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_1.npz
19:26:01 [INFO] Fold 1 | X_train: (2206, 217), X_test: (267, 217) | scale_features=True | y_mean=6.721 y_std=0.867
19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_2.npz


19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_2.npz
19:26:01 [INFO] Fold 2 | X_train: (2128, 217), X_test: (267, 217) | scale_features=True | y_mean=6.736 y_std=0.840
19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_3.npz
19:26:01 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_3.npz
19:26:01 [INFO] Fold 3 | X_train: (2257, 217), X_test: (262, 217) | scale_features=True | y_mean=6.753 y_std=0.857


In [ ]:
logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP DRD2 Lo — rdkit_desc")

19:26:01 [INFO] Estimated time: ~45 min
19:26:01 [INFO] 
OUTER FOLD 1
19:26:05 [INFO]   [1/150] inner MSE=0.4739 (score=-0.4739) (4s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
19:26:10 [INFO]   [2/150] inner MSE=0.5610 (score=-0.5610) (5s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
19:26:12 [INFO]   [3/150] inner MSE=0.5866 (score=-0.5866) (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
19:26:16 [INFO]   [4/150] inner MSE=0.5332 (score=-0.5332) (4s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
19:26:17 [INFO]   [5/150] inner MSE=0.5494 (score=-0.5494) (1s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
19:26:21 [INFO]   [6/150] inner MSE=0.7344 (score=-0.7344) (4s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
19:26:27 [INFO]   [7/150] inner MSE=0.7532 (score=-0.7532) (5s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
19:26:32 [INFO]   [8/150] inner MSE=1.8776 (score=-1.8776) (5s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
19:26:35 [INFO]   [9/150] inner MSE=365.1653 (score=-365.1653) (3s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
1